# Notebook 5: Deploy the Foodly AI Agent

## What This Notebook Does
Deploys the Foodly Support Agent to production:
1. **Log** the agent as an MLflow model (packages code + dependencies)
2. **Validate** with pre-deployment testing
3. **Register** in Unity Catalog (versioned model registry)
4. **Deploy** to Model Serving (REST API endpoint)

## Deployment Flow
```
main.py + helpers.py
     |
     v
MLflow Log Model --> MLflow Run (artifacts stored)
     |
     v
Pre-deployment Validation (test with sample input)
     |
     v
Register in Unity Catalog --> agents.main.foodly-ai-assistant (v1, v2...)
     |
     v
Deploy to Model Serving --> REST API Endpoint (auto-scaling)
     |
     v
Chat UI / API Consumers can now call the agent
```

**Run this notebook in Databricks after notebooks 1-4**

## Step 1: Log the Model with MLflow

This packages:
- `main.py` - The agent entry point
- `helpers.py` - LangGraph-to-MLflow bridge
- All Databricks resource dependencies (for automatic auth passthrough)
- Python package versions (for reproducibility)

In [ ]:
from main import UC_TOOL_NAMES, VECTOR_SEARCH_TOOL

import mlflow
from mlflow.models.resources import DatabricksFunction
from pkg_resources import get_distribution

resources = []

# Add Vector Search resources (index + endpoint)
resources.extend(VECTOR_SEARCH_TOOL.resources)

# Add UC Function resources
for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

print(f"Resources for auth passthrough: {len(resources)}")
for r in resources:
    print(f"  - {r}")

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="foodly-ai-assistant",
        code_paths=["helpers.py"],
        python_model="main.py",
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ],
        resources=resources,
    )

## Step 2: Pre-deployment Validation

Test the packaged model in an isolated environment before deploying.

In [ ]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/foodly-ai-assistant",
    input_data={"input": [{"role": "user", "content": "Tell me about foodly return policy"}]},
    env_manager="uv",
)

## Step 3: Register in Unity Catalog

This creates a versioned model in Unity Catalog, enabling:
- Model versioning (rollback to previous versions)
- Access control (who can use/deploy the model)
- Lineage tracking (what data/code produced this model)

In [ ]:
mlflow.set_registry_uri("databricks-uc")

# TODO: Update these to match your Unity Catalog setup
catalog = "agents"
schema = "main"
model_name = "foodly-ai-assistant"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri,
    name=UC_MODEL_NAME,
)
print(f"Registered model: {UC_MODEL_NAME} version {uc_registered_model_info.version}")

## Step 4: Deploy to Model Serving

This creates a serverless REST API endpoint that:
- Auto-scales based on traffic
- Can scale to zero when idle (cost saving)
- Has built-in authentication
- Supports streaming responses

In [ ]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    scale_to_zero=True,
    tags={
        "name": "foodly-ai-assistant-endpoint",
        "creator": "your-name",
    },
)

## Next Steps

After deployment:
1. **Test** in AI Playground (Databricks UI)
2. **Connect** the Streamlit Chat UI (see `app.py`)
3. **Monitor** via MLflow traces and serving metrics
4. **Iterate** - improve prompts, add tools, deploy new versions